# 리포트 논리·설계 검토 — 주장 범위와 실험이 결정할 것

> ### 한 일
> **현행 계획과 보고서의 주장·원장·생성 코드를 대조하고 순위와 해석식 반례를 재계산했다.**

### 결과
1. 정량 재계산과 근거 대조 6 [^1]항목이 빌더의 명시한 조건을 통과했다.
2. 공유 문턱·위치 복원·파형 독립성·기체 순위의 일부 문장은 근거 범위보다 크다.
3. 자유공간 비교에서 실외 배치·추적 성능으로 넘어가는 구간에는 별도 평가가 필요하다.
4. 원장에 이미 적힌 제한과 새 실험에서 정할 조건을 구분했다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 직접 확인 | 현행 계획·선별 보고서·빌더·원장의 대조와 CPU 해석 계산 |
| 검토 범위 | 실측·문헌 전체 조사·GPU 재실행은 범위 밖이다. 합성 반례는 가정의 범위를 확인한다. |

### 재현

```bash
/workspace/.venvs/py312/bin/python benchmark/review_research_logic_0916.py
```

| | |
|---|---|
| 출력 | `outputs/research_logic_review_0916.json` |
| 소요 | CPU 단일 코어의 경량 계산 |

---

## 전체 흐름에 대한 판단

현재 자료는 산란·수치 동작을 자세히 설명한다. 다음 연구에서는 그 관찰이 어떤 수신 처리·배치 결정을 바꾸며, 실제 탐지와 추적에 무엇을 더하는지 연결해야 한다.

권장 흐름: 해결할 실패 상황 → 원인 후보 → 통제 실험 → 사용할 방법 → 같은 자원의 비교 → 별도 세션에서의 평가.

새 기능의 수와 연구 기여는 구분한다. 아래의 확정 문장 오류는 수정 대상으로, 설계 위험은 앞으로 가를 가설로 읽는다.

## 문턱 공유는 실험 결과와 가정을 나눠 써야 한다

**구분:** 확인된 제목과 본문의 범위 차이

**관찰:** 공유 문턱의 출처는 W1 [^2]다. NR은 검사 칸 5 [^3]개 중 5 [^4]개가 건너뛰어져 자기 문턱이 비어 있다.

**해석:** 본문은 W1 문턱을 함께 쓰는 선택이라고 설명하지만 제목은 여러 밴드에서 같은 문턱을 발견한 것처럼 읽힌다.

**권고:** 제목을 공유 문턱 가정의 영향 평가로 바꾸고, 각 파형의 유효 Doppler 축에서 별도 교정·평가를 한다. 겹치는 신뢰구간과 동등성 입증도 구분한다.

근거: [src/build_part10_results.py](../src/build_part10_results.py) 행 753 [^5] · [src/build_part10_results.py](../src/build_part10_results.py) 행 794 [^6]

## 수신기 추가와 전역 위치 복원은 다른 주장이다

**구분:** 기하 반례와 코드로 확인

**관찰:** 합성 배치에서 국소 상태 랭크는 6 [^7]이며, 거울상 궤적 간 위치 간격은 8 [^8] m다. 그런데 거리 차는 3.553e-15 [^9] m, Doppler 차는 3.553e-15 [^10] Hz다.

**해석:** 이 반례는 수신기 개수나 국소 full rank만으로 전역 유일성을 보장할 수 있는지를 시험한다. 실제 보고서 기하의 제약은 별도이며, pinv가 버린 영공간의 불확실성을 유한 RMS로 읽어서도 안 된다.

**권고:** 관측시간·운동모델·센서 배치·허용 높이·가시영역을 붙여 표현하고, 허용 상태 전체에서 모호성을 확인한다. CRLB와 실제 추적 오차를 별도로 보고한다.

근거: [src/build_part09_detector.py](../src/build_part09_detector.py) 행 707 [^11] · [benchmark/verify_observability.py](../benchmark/verify_observability.py) 행 341 [^12] · [benchmark/verify_observability.py](../benchmark/verify_observability.py) 행 627 [^13]

## 기체 순위와 상관 해석은 원장 수치에 맞춰 고쳐야 한다

**구분:** 직접 재계산한 서술 오류

**관찰:** 현재 원장의 5 [^14]기체에서 mini5pro는 단일 자세 1 [^15]위, 자세평균 4 [^16]위다. Pearson 상관은 크기 -0.447 [^17], σ 산포 -0.694 [^18]다.

**해석:** 양쪽에서 가장 견고하다는 문장과 크기 쪽 상관이 더 강하다는 문장이 숫자와 어긋난다. 후자는 Pearson 절댓값 비교이며 인과관계는 별도 실험의 대상이다.

**권고:** 단일 자세와 자세평균 순위를 각각 계산해 문장을 생성한다. 보고서와 원장 생성기의 산문을 함께 수정해 재빌드에도 같은 범위를 유지한다.

근거: [src/build_part10_results.py](../src/build_part10_results.py) 행 1282 [^19] · [benchmark/sigma_sensitivity.py](../benchmark/sigma_sensitivity.py) 행 705 [^20] · [src/build_part05_anchor.py](../src/build_part05_anchor.py) 행 948 [^21]

## 알려진 OFDM 심볼로 나눠도 잡음 특성은 데이터에 의존한다

**구분:** 해석식으로 확인한 과한 표현

**관찰:** 동일 평균 심볼 에너지의 QPSK와 16-QAM을 열거하면 E[|X|⁻²]는 각각 1.000 [^22]와 1.889 [^23]다. 단순 나눗셈 후 평균 잡음 차는 2.762 [^24] dB다.

**해석:** Y=HX+N에서 Y/X=H+N/X다. 잡음 없는 대각 채널 성분의 복원과 실제 RD·검출 성능의 데이터 독립성은 다르다. 이 수치는 검출 손실을 측정한 값이 아니다.

**권고:** 문장을 잡음 없는 대각 채널로 한정한다. 같은 채널·잡음·자원에서 payload와 변조를 바꾸고, ICI·clipping·가중 추정의 영향을 별도 평가한다. 계획 표에는 이미 관련 경고가 있다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 48 [^25]

## 모든 지면 장면이라는 표현에 조준 안테나 반례가 있다

**구분:** 기존 원장으로 확인한 범위 오류

**관찰:** 조준 안테나를 쓴 지면 묶음은 4 [^26]칸이며 고립 자세가 있는 칸은 1 [^27]칸이다.

**해석:** 영문 계획의 every ground scene은 등방 조건에서 본 현상을 조준 안테나 조건까지 넓힌다. 한국어 초안은 조준 조건의 예외를 이미 설명한다.

**권고:** 장면·안테나·높이·표본 범위를 같은 문장에 붙인다. 다른 조건을 합친 전칭 문장을 사용하기 전에 원장 묶음을 다시 센다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 84 [^28]

## 자유공간 교차 비교 뒤에 실외 예측력을 따로 평가해야 한다

**구분:** 아직 닫히지 않은 실험 설계

**관찰:** 계획은 커널과의 자유공간 비교 뒤 PathSolver로 환경을 모델링한다. 사전 허용 오차는 현재 positioning builder의 추가 작업에 적혀 있다.

**해석:** 자유공간 일치는 그 조건의 일관성을 보여 준다. 지면·건물·안테나를 넣은 뒤 배치 순위와 추적 누락까지 예측하는지는 별도 문제다. 현재 규약도 두 근사의 일관성과 실측 타당성을 구분한다.

**권고:** 설정 선택에 쓸 기하와 평가 기하를 분리하고 상대 지표·허용 오차를 먼저 고정한다. 선택한 설정을 고정한 채 지면·벽·실제 비행의 배치 순위 및 실패를 평가한다.

근거: [benchmark/isac_plan_positioning_0915.py](../benchmark/isac_plan_positioning_0915.py) 행 1498 [^29] · [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 167 [^30]

## 고립 자세 대체는 원인 확인과 분리해야 한다

**구분:** 원인 추론 및 전처리 선택의 위험

**관찰:** 고립 자세를 복소 중앙값으로 바꾸면 빗살 대비가 커지고, 같은 수의 비고립 자세를 바꾸는 대조에서는 효과가 작다. 현재 원장은 원인 미확정을 명시한다.

**해석:** 큰 편차를 골라 지웠다는 대조는 그 자세가 지표를 좌우함을 보여 준다. 수치 오류·물리적인 flash·실제 장애 중 무엇을 제거했는지는 추가 근거가 필요하다. 대비 회복도 탐지·추적 회복과 별도다.

**권고:** 원자료 결과를 유지하고 대체 결과는 민감도로 병기한다. 경로 목록, packet loss, gain 및 clipping 기록으로 사유를 확인하고, 규칙을 고정한 별도 세션에서 실패율까지 평가한다.

근거: [benchmark/isac_plan_corpus_0915.py](../benchmark/isac_plan_corpus_0915.py) 행 1438 [^31] · [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 84 [^28]

## 각도·높이 관측과 수신 채널 배분을 먼저 고정해야 한다

**구분:** 현행 계획 간 조건 불일치

**관찰:** 계획의 차별점은 네 RX 각도를 말하지만 배분은 감시 RX 세 개와 기준 또는 통신 RX 한 개다. 캠페인의 초기 상태는 알려진 표적 높이를 가정한다.

**해석:** sector amplitude ratio에서 azimuth로 가는 보정 관계와 불확실성이 필요하다. RTK 높이를 수신 처리에 주면 보조 정보가 있는 추적이 된다. 기존 캠페인은 truth를 평가에만 사용하도록 규정한다.

**권고:** 보유 안테나의 실제 배치·패턴을 정하고 별도 위치에서 각도 오차를 평가한다. 알려진 높이 실험과 미지 높이 실험을 분리하고, oracle angle은 상한 대조로 표시한다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 141 [^32] · [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 151 [^33] · [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 158 [^34] · [benchmark/design_isac_campaign_0915.py](../benchmark/design_isac_campaign_0915.py) 행 112 [^35] · [benchmark/design_isac_campaign_0915.py](../benchmark/design_isac_campaign_0915.py) 행 115 [^36]

## 디지털 트윈의 기여를 같은 하드웨어에서 가려야 한다

**구분:** 비교군 설계가 더 필요함

**관찰:** 제시된 비교군은 단일 무지향 안테나, sector 제거, 트윈 없는 배치다.

**해석:** 안테나 패턴·채널 수·개구·커버리지까지 바뀌면 효과를 트윈의 배치 판단에 귀속하기 어렵다. 트윈 없는 배치도 선택 규칙을 고정해야 비교가 재현된다.

**권고:** 같은 안테나·채널·설치 후보·RF 예산에서 고정 배치, 기하 휴리스틱, 같은 보정 예산의 실측 탐색, 트윈 배치를 비교한다. 선정 뒤 별도 비행·날짜에서 평가한다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 167 [^30] · [benchmark/design_isac_waveform_benchmark_0915.py](../benchmark/design_isac_waveform_benchmark_0915.py) 행 73 [^37]

## ISAC 자원 절충에는 송신 측 개입이 필요하다

**구분:** 계획은 인식하지만 실행 계약은 미정

**관찰:** 모든 알려진 payload를 sensing에 쓰면서 revisit를 바꾸는 설계가 있다. 캠페인은 처리 창 변경만으로 송신 overhead가 생기는 것은 아니라고 명시한다.

**해석:** 같은 송신 IQ를 재처리하는 실험은 계산·추적 정책 평가다. 통신과 sensing의 RF 자원 경쟁을 주장하려면 실제 바뀐 송신 자원이 있어야 한다. 같은 X410의 통신 RX는 공유 clock 조건의 실험이다.

**권고:** 송신 IQ 고정·수신 처리만 변경한 대조를 먼저 둔다. 이후 airtime·전력·자원 배치 중 개입할 것을 정하고 동일 offered load에서 goodput·지연·추적 연속성을 비교한다.

근거: [benchmark/design_isac_campaign_0915.py](../benchmark/design_isac_campaign_0915.py) 행 137 [^38] · [benchmark/design_isac_waveform_benchmark_0915.py](../benchmark/design_isac_waveform_benchmark_0915.py) 행 59 [^39] · [benchmark/design_isac_waveform_benchmark_0915.py](../benchmark/design_isac_waveform_benchmark_0915.py) 행 73 [^37]

## 로터 확인은 실패한 후보와 비표적도 포함해야 한다

**구분:** 선택 편향 및 오경보 단위의 위험

**관찰:** 계획은 이미 확인된 track에 blade line이 있는지 보고 false track 감소를 평가하려 한다. 위치 bin과 동체 Doppler도 tracker에서 받는다.

**해석:** 살아남은 track만 고르면 로터 관측이 어려운 사례가 빠진다. 다중 분류를 제외해도 false track 감소 주장은 움직이는 clutter와 비표적 후보를 요구한다.

**권고:** 같은 후보 목록에 로터 사용·미사용을 적용하고 관측시간과 확인 지연을 맞춘 persistence/SNR baseline을 둔다. 참 track 거부·확인 시간·시간당 거짓 track을 함께 보고, 비행·날짜 단위로 나눠 평가한다.

근거: [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 128 [^40] · [benchmark/design_isac_campaign_0915.py](../benchmark/design_isac_campaign_0915.py) 행 129 [^41]

## 표본 수 계산의 조건

오경보 목표 0.001 [^42]·단측 신뢰수준 0.95 [^43]에서 무오경보 상한을 계산하면 독립 H0 시행 2995 [^44]회가 필요하다.

이 값은 캠페인 원장의 이항 계산을 재현한 것이다. 연속 프레임을 그 수만큼 모으는 것과 독립 시행 확보는 다르며, cell·frame·track의 오경보 단위도 구분한다.

환경과 궤적을 보고 문턱을 정한 데이터는 최종 성능 평가에서 분리한다. 세션 단위 오차와 실패 구간 길이도 함께 보고한다.

## 기술 조합의 희소성에서 연구 기여로 바로 넘어가지 않기

**구분:** 연구 구성에 대한 판단

**관찰:** 기존 보고서는 드론 메쉬·특정 엔진·진폭 대조·게재 여부의 교집합을 조사한다. 새 계획에는 각도·다중 표적·통신 지표·트윈 배치가 차별점으로 나열된다.

**해석:** 같은 구현 조합을 찾지 못했다는 사실과 중요한 미해결 문제를 해결했다는 사실은 다른 근거가 필요하다. 이번 검토는 문헌 전체의 신규성을 판정한 것이 아니다.

**권고:** 주요 질문을 먼저 정한다. 예를 들어 같은 안테나와 통신 자원에서 트윈이 고른 배치가 기하 휴리스틱·실측 탐색보다 별도 비행의 track 단절을 줄이는지 시험할 수 있다. 나머지 기능은 그 질문의 근거로 배치한다.

근거: [src/build_part02_prior_work.py](../src/build_part02_prior_work.py) 행 655 [^45] · [docs/MOBICOM_PIPELINE_PLAN_0916.md](../docs/MOBICOM_PIPELINE_PLAN_0916.md) 행 141 [^32]

## 이미 적절히 제한된 부분

- 현재 한국어 계획은 조준 전후 대비의 원인을 미확정으로 남겨 두었다.
- hover·단일 거리·합쳐진 채널 자료로 추적 정확도를 얻을 수 없다는 범위를 적었다.
- 캠페인에는 truth 분리, holdout, 독립 H0 조건, 동일 자원 비교가 이미 들어 있다.
- 커널과 PathSolver의 일치는 실측 타당성 대조와 구분돼 있다.

이 항목들은 이미 마련된 제한으로 분류했다. 짧은 계획·제목·실행 코드까지 같은 조건을 유지하는 것이 과제다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 문턱·순위·OFDM·지면·관측가능성 문장 범위를 바로잡는다 | 독자가 실제 계산 범위를 읽게 된다 | 해당 원장 생성기와 report builder |
| 주요 연구 질문과 비교할 배치·수신 정책을 고정한다 | 무엇이 연구 기여인지 평가할 기준이 정해진다 | 현행 pipeline plan |
| 안테나·채널·높이·통신 자원 계약을 고정한다 | 실험이 실제로 제공할 관측과 자원이 정해진다 | campaign 및 waveform design |
| 보정 데이터와 평가 비행을 나누고 실패 사례까지 보고한다 | 설정 선택 이후 새 조건의 예측력을 평가한다 | 별도 날짜·비행·장면의 측정 계획 |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 45개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/research_logic_review_0916.json` | `check_count` | 6 |
| [^2] | `outputs/research_logic_review_0916.json` | `threshold.snr90_source_mode` | W1 |
| [^3] | `outputs/research_logic_review_0916.json` | `threshold.g1_total_cells` | 5 |
| [^4] | `outputs/research_logic_review_0916.json` | `threshold.g1_skipped_cells` | 5 |
| [^5] | `outputs/research_logic_review_0916.json` | `sources.threshold.line` | 753 |
| [^6] | `outputs/research_logic_review_0916.json` | `sources.threshold_gap.line` | 794 |
| [^7] | `outputs/research_logic_review_0916.json` | `observability.local_rank` | 6 |
| [^8] | `outputs/research_logic_review_0916.json` | `observability.position_separation_m` | 8 |
| [^9] | `outputs/research_logic_review_0916.json` | `observability.max_range_difference_m` | 3.553e-15 |
| [^10] | `outputs/research_logic_review_0916.json` | `observability.max_doppler_difference_hz` | 3.553e-15 |
| [^11] | `outputs/research_logic_review_0916.json` | `sources.observability.line` | 707 |
| [^12] | `outputs/research_logic_review_0916.json` | `sources.gramian.line` | 341 |
| [^13] | `outputs/research_logic_review_0916.json` | `sources.pinv.line` | 627 |
| [^14] | `outputs/research_logic_review_0916.json` | `ranking.n_airframes` | 5 |
| [^15] | `outputs/research_logic_review_0916.json` | `ranking.mini_single_rank` | 1 |
| [^16] | `outputs/research_logic_review_0916.json` | `ranking.mini_average_rank` | 4 |
| [^17] | `outputs/research_logic_review_0916.json` | `ranking.corr_extent` | -0.4475 |
| [^18] | `outputs/research_logic_review_0916.json` | `ranking.corr_spread` | -0.6939 |
| [^19] | `outputs/research_logic_review_0916.json` | `sources.rank.line` | 1282 |
| [^20] | `outputs/research_logic_review_0916.json` | `sources.rank_generator.line` | 705 |
| [^21] | `outputs/research_logic_review_0916.json` | `sources.correlation.line` | 948 |
| [^22] | `outputs/research_logic_review_0916.json` | `ofdm.qpsk_inverse_energy` | 1 |
| [^23] | `outputs/research_logic_review_0916.json` | `ofdm.qam_inverse_energy` | 1.889 |
| [^24] | `outputs/research_logic_review_0916.json` | `ofdm.noise_ratio_db` | 2.762 |
| [^25] | `outputs/research_logic_review_0916.json` | `sources.ofdm.line` | 48 |
| [^26] | `outputs/research_logic_review_0916.json` | `aimed_ground.n_cells` | 4 |
| [^27] | `outputs/research_logic_review_0916.json` | `aimed_ground.n_cells_with_outliers` | 1 |
| [^28] | `outputs/research_logic_review_0916.json` | `sources.all_ground.line` | 84 |
| [^29] | `outputs/research_logic_review_0916.json` | `sources.admission.line` | 1498 |
| [^30] | `outputs/research_logic_review_0916.json` | `sources.placement.line` | 167 |
| [^31] | `outputs/research_logic_review_0916.json` | `sources.replacement.line` | 1438 |
| [^32] | `outputs/research_logic_review_0916.json` | `sources.scope.line` | 141 |
| [^33] | `outputs/research_logic_review_0916.json` | `sources.channels.line` | 151 |
| [^34] | `outputs/research_logic_review_0916.json` | `sources.bearing.line` | 158 |
| [^35] | `outputs/research_logic_review_0916.json` | `sources.known_height.line` | 112 |
| [^36] | `outputs/research_logic_review_0916.json` | `sources.truth.line` | 115 |
| [^37] | `outputs/research_logic_review_0916.json` | `sources.fairness.line` | 73 |
| [^38] | `outputs/research_logic_review_0916.json` | `sources.policy.line` | 137 |
| [^39] | `outputs/research_logic_review_0916.json` | `sources.shared_clock.line` | 59 |
| [^40] | `outputs/research_logic_review_0916.json` | `sources.rotor.line` | 128 |
| [^41] | `outputs/research_logic_review_0916.json` | `sources.negative_trials.line` | 129 |
| [^42] | `outputs/research_logic_review_0916.json` | `false_alarm.target` | 0.001 |
| [^43] | `outputs/research_logic_review_0916.json` | `false_alarm.confidence` | 0.95 |
| [^44] | `outputs/research_logic_review_0916.json` | `false_alarm.independent_trials` | 2995 |
| [^45] | `outputs/research_logic_review_0916.json` | `sources.novelty.line` | 655 |